# Body Performance — Classification Notebook
## Predicting Fitness Grade (A / B / C / D)

**Dataset:** bodyPerformance.csv | **Task:** Multi-class Classification | **Target:** `class`

---
| Section | Topic |
|---------|-------|
| 1.1 | Dataset Overview |
| 1.2 | Column Understanding |
| 1.3 | Data Type Verification |
| 1.4 | Missing Values Analysis |
| 1.5 | Duplicate Detection |
| 1.6 | Data Validity Checks |
| 1.7 | Univariate Analysis |
| 1.8 | Distribution Analysis |
| 1.9 | Outlier Detection |
| 1.10 | Correlation Analysis |
| 1.11 | Final EDA Summary |
| 2 | Feature Engineering |
| 3 | Model Building & Evaluation |


## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
sns.set_theme(style='whitegrid', palette='husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
RANDOM_STATE = 42
print("All libraries loaded successfully.")

---
## 1.1 Dataset Overview

First look at the dataset size, shape, and sample contents.

In [ ]:
df_raw = pd.read_csv('bodyPerformance.csv')
df     = df_raw.copy()

n_rows, n_cols = df.shape
print(f"Number of Rows    : {n_rows:,}")
print(f"Number of Columns : {n_cols}")
print()
print("Column Names:")
for idx, col in enumerate(df.columns, 1):
    print(f"  {idx:>2}. {col}")

In [ ]:
print("Sample rows (first 5):")
display(df.head(5))
print("\nRandom sample (5 rows):")
display(df.sample(5, random_state=42))

### Dataset Description

The **Body Performance Dataset** contains fitness measurements from **13,393 individuals** across **12 columns**.
Each row represents one person who completed a standardised physical fitness assessment.
The dataset captures demographics (age, gender), body composition (height, weight, body fat),
cardiovascular indicators (blood pressure), and four performance tests (grip force, flexibility,
sit-up endurance, and broad jump distance). The target variable `class` assigns each individual
a fitness grade from **A** (highest) to **D** (lowest fitness). Classes are near-perfectly balanced
at ~25% each, making this a clean multi-class classification problem with no imbalance issues.


---
## 1.2 Column Understanding

Each column is described with meaning, expected type, and validation rules.

In [ ]:
col_info = pd.DataFrame({
    'Column': ['age','gender','height_cm','weight_kg','body fat_%',
               'diastolic','systolic','gripForce',
               'sit and bend forward_cm','sit-ups counts','broad jump_cm','class'],
    'Meaning': [
        'Age of participant in years',
        'Biological sex: Male (M) or Female (F)',
        'Height in centimetres',
        'Body weight in kilograms',
        'Percentage of body mass composed of fat tissue',
        'Diastolic blood pressure (mmHg) — heart at rest',
        'Systolic blood pressure (mmHg) — heart contracting',
        'Hand grip strength measured by dynamometer (kg)',
        'Sit-and-reach flexibility test result (cm)',
        'Number of sit-ups completed in timed test',
        'Standing broad jump distance (cm)',
        'Fitness grade: A=best, B, C, D=lowest'
    ],
    'Expected Type': ['Numeric','Categorical','Numeric','Numeric','Numeric',
                      'Numeric','Numeric','Numeric','Numeric','Numeric','Numeric','Categorical'],
    'Validation Rules': [
        '21-64 (adult active range in dataset)',
        'M or F only',
        '> 100 cm',
        '> 0 kg, typically 30-200',
        '3-60 % (healthy adult range)',
        '40-130 mmHg',
        '60-200 mmHg',
        '> 0 kg (cannot be negative)',
        '-30 to 80 cm (human anatomical range)',
        '0-80 (test maximum)',
        '0-400 cm (world record ~343 cm)',
        'A, B, C, or D only'
    ]
})
display(col_info)

---
## 1.3 Data Type Verification

Verify whether each column has the correct data type. Numeric columns stored as text break calculations.

In [ ]:
dtype_df = df.dtypes.rename('Detected Type').to_frame()
dtype_df['Expected Type'] = ['Numeric','Categorical','Numeric','Numeric','Numeric',
                              'Numeric','Numeric','Numeric','Numeric','Numeric','Numeric','Categorical']
dtype_df['Match'] = dtype_df.apply(
    lambda r: 'Correct' if (
        (r['Expected Type']=='Numeric'     and r['Detected Type'] in ['float64','int64']) or
        (r['Expected Type']=='Categorical' and str(r['Detected Type']) in ['object','str','string'])
    ) else 'MISMATCH', axis=1)
display(dtype_df)

print()
print("Findings:")
print("  - All 10 numeric columns are correctly stored as float64.")
print("  - 'age' and 'sit-ups counts' are float64 but represent whole numbers.")
print("    This is acceptable and requires no conversion.")
print("  - 'gender' and 'class' are stored as object/str — correct for categorical data.")
print("  - No type mismatches requiring mandatory correction were identified.")

---
## 1.4 Missing Values Analysis

Identify, quantify, and resolve all missing data before modelling.

In [ ]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(4)

missing_df = pd.DataFrame({
    'Missing Count': null_counts,
    'Missing %': null_pct
})

def assign_strategy(col, pct, series):
    if pct == 0:
        return 'None needed'
    elif pct < 5:
        return 'Drop rows (< 5% impact)'
    elif pct < 30:
        if series.dtype == 'object':
            return 'Fill with Mode (categorical)'
        elif abs(series.skew()) > 1:
            return 'Fill with Median (skewed numeric)'
        else:
            return 'Fill with Mean (normal numeric)'
    else:
        return 'Drop column (> 30% missing)'

missing_df['Strategy'] = [
    assign_strategy(col, null_pct[col], df[col]) for col in df.columns
]
display(missing_df)

print()
if null_counts.sum() == 0:
    print("RESULT: No missing values detected in any column.")
    print("        No imputation or row removal required.")
else:
    print("Applying strategies...")
    for col in missing_df.index:
        strat = missing_df.loc[col, 'Strategy']
        if 'Drop rows' in strat:
            df.dropna(subset=[col], inplace=True)
        elif 'Median' in strat:
            df[col].fillna(df[col].median(), inplace=True)
        elif 'Mean' in strat:
            df[col].fillna(df[col].mean(), inplace=True)
        elif 'Mode' in strat:
            df[col].fillna(df[col].mode()[0], inplace=True)
        elif 'Drop column' in strat:
            df.drop(columns=[col], inplace=True)
    print(f"Shape after handling: {df.shape}")

---
## 1.5 Duplicate Detection

Duplicate rows inflate patterns and can cause data leakage between train and test sets.

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Number of exact duplicate rows found: {n_dupes}")

if n_dupes > 0:
    print("\nThe duplicate row(s):")
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))
    df.drop_duplicates(inplace=True)
    print(f"\nAction: Duplicates removed.")
    print(f"New shape: {df.shape}")
else:
    print("No duplicate rows found. No action required.")

---
## 1.6 Data Validity Checks

Detect values that are non-null but physiologically or logically impossible.

In [ ]:
print("=== Validity Check Report ===")
print()

checks = [
    ('systolic',                df['systolic'] <= 40,
     'Systolic BP <= 40 mmHg (impossible for living person)'),
    ('diastolic',               df['diastolic'] <= 40,
     'Diastolic BP <= 40 mmHg (impossible for living person)'),
    ('body fat_%',              df['body fat_%'] > 70,
     'Body fat > 70% (physiologically extreme)'),
    ('sit and bend forward_cm', df['sit and bend forward_cm'] > 100,
     'Sit-and-reach > 100 cm (beyond human anatomical limit)'),
    ('gripForce',               df['gripForce'] == 0,
     'Grip force = 0 kg (likely measurement/recording error)'),
    ('broad jump_cm',           df['broad jump_cm'] == 0,
     'Broad jump = 0 cm (possibly no attempt or error)'),
    ('weight_kg',               df['weight_kg'] < 0,
     'Negative weight (physically impossible)'),
    ('gender',                  ~df['gender'].isin(['M','F']),
     "Gender label not in ['M','F']"),
    ('class',                   ~df['class'].isin(['A','B','C','D']),
     "Class label not in ['A','B','C','D']"),
]

for col, mask, description in checks:
    count = int(mask.sum())
    status = f"VIOLATION ({count} rows)" if count > 0 else "OK"
    print(f"  [{status:<22}]  {description}")

print()
print("=== Action Plan ===")
print("1. REMOVE rows with systolic or diastolic BP <= 40 (medically impossible).")
print("2. CAP sit-and-reach at physical limits (Winsorise) — likely input errors.")
print("3. RETAIN grip=0 and broad jump=0 — could represent genuine test failures.")
print("4. No negative weights, no invalid gender/class labels found.")

In [ ]:
before = len(df)

# Fix 1: Remove impossible blood pressure values
df = df[(df['systolic'] > 40) & (df['diastolic'] > 40)]
removed_bp = before - len(df)
print(f"Rows removed (BP <= 40 mmHg) : {removed_bp}")

# Fix 2: Cap sit-and-reach at physiological limits
old_max = df['sit and bend forward_cm'].max()
old_min = df['sit and bend forward_cm'].min()
df['sit and bend forward_cm'] = df['sit and bend forward_cm'].clip(lower=-25, upper=80)
print(f"sit and bend forward_cm capped: [{old_min:.1f}, {old_max:.1f}] -> [-25, 80]")

print(f"\nDataset shape after validity fixes: {df.shape}")

---
## 1.7 Univariate Analysis

Core descriptive statistics for every numeric column.

In [ ]:
NUM_COLS = ['age','height_cm','weight_kg','body fat_%','diastolic',
            'systolic','gripForce','sit and bend forward_cm',
            'sit-ups counts','broad jump_cm']

stats_table = df[NUM_COLS].agg(['mean','median','std','min','max']).T
stats_table.columns = ['Mean','Median','Std Dev','Min','Max']
stats_table['Skewness'] = df[NUM_COLS].skew().round(3)
stats_table = stats_table.round(3)

print("Descriptive Statistics Summary:")
display(stats_table)

In [ ]:
print("=== Column-by-Column Interpretation ===")
print()
interpretations = [
    ('age',                     'Mean=36.8yr. Right-skewed (0.60): more young adults. Range 21-64 confirms adult dataset.'),
    ('height_cm',               'Mean=168.6cm. Near-symmetric (-0.19). Bimodal in reality (male/female peaks).'),
    ('weight_kg',               'Mean=67.1kg. Slight right skew (0.35): a few heavier individuals pull the tail.'),
    ('body fat_%',              'Mean=24.0%. Right-skewed (0.36): most 10-40%, some extreme high-fat outliers.'),
    ('diastolic',               'Mean=80.1 mmHg. Near-symmetric. Clinically normal resting pressure.'),
    ('systolic',                'Mean=130.3 mmHg. Near-symmetric. Slightly elevated; some hypertensive participants.'),
    ('gripForce',               'Mean=36.9 kg. Near-symmetric (0.02). Bimodal: male/female subpopulations.'),
    ('sit and bend forward_cm', 'Mean=15.7cm after capping. Moderate right skew (0.79). Good spread across flexibility levels.'),
    ('sit-ups counts',          'Mean=39.8 reps. Slight left skew (-0.47). Performance ceiling at 80 observed.'),
    ('broad jump_cm',           'Mean=190cm. Slight left skew (-0.42). Large Std=39.9 reflects wide fitness variation.'),
]
for col, note in interpretations:
    print(f"  {col:<30}: {note}")

---
## 1.8 Distribution Analysis

Histograms with KDE reveal shape, skewness, bimodality, and anomalies.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(NUM_COLS):
    ax = axes[idx]
    series = df[col].dropna()
    skew_val = series.skew()

    sns.histplot(series, kde=True, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(series.mean(),   color='red',    ls='--', lw=1.5, label=f'Mean={series.mean():.1f}')
    ax.axvline(series.median(), color='orange', ls='--', lw=1.5, label=f'Median={series.median():.1f}')

    if abs(skew_val) > 1:
        shape_label = 'Highly Skewed'
    elif abs(skew_val) > 0.5:
        shape_label = 'Moderately Skewed'
    else:
        shape_label = 'Approximately Normal'

    ax.set_title(f'{col}\nSkew={skew_val:.2f} | {shape_label}', fontsize=8, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_xlabel('')

# Target class distribution
ax = axes[len(NUM_COLS)]
class_counts = df['class'].value_counts().sort_index()
bars = ax.bar(class_counts.index, class_counts.values,
              color=sns.color_palette('Set2', 4), edgecolor='white')
ax.set_title('Target: class Distribution', fontsize=9, fontweight='bold')
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=8)

for j in range(len(NUM_COLS)+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribution Analysis — All Numeric Variables', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("=== Distribution Findings ===")
print("  age             : Right-skewed — more younger participants in the sample.")
print("  height_cm       : Bimodal — two peaks corresponding to male/female subgroups.")
print("  weight_kg       : Right-skewed — tail of heavier individuals; possible outliers.")
print("  body fat_%      : Right-skewed — most in healthy range; a few extreme values.")
print("  diastolic       : Normal — symmetric bell shape centred at ~80 mmHg.")
print("  systolic        : Normal — centred ~130 mmHg, slight right tail.")
print("  gripForce       : Bimodal — reflects distinct male/female grip strength distributions.")
print("  sit&bend fwd    : Moderately right-skewed — capping resolved extreme tail.")
print("  sit-ups         : Slightly left-skewed — moderate performers most frequent.")
print("  broad jump      : Left-skewed — performance clusters around 190 cm.")
print("  class (target)  : Balanced — ~25% per class; no imbalance correction needed.")

---
## 1.9 Outlier Detection

Boxplots identify outliers using the IQR rule (1.5 x IQR fence).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

outlier_records = []
for idx, col in enumerate(NUM_COLS):
    ax = axes[idx]
    series = df[col].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = int(((series < lo) | (series > hi)).sum())
    pct = n_out / len(series) * 100

    sns.boxplot(y=series, ax=ax, color='lightcoral',
                flierprops={'markersize': 3, 'alpha': 0.5})
    ax.set_title(f'{col}\n{n_out} outliers ({pct:.1f}%)', fontsize=8, fontweight='bold')
    outlier_records.append({'Column': col, 'N Outliers': n_out, '%': round(pct,2),
                            'Lower Fence': round(lo,2), 'Upper Fence': round(hi,2)})

plt.suptitle('Outlier Detection — Boxplots (IQR Rule)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

out_df = pd.DataFrame(outlier_records)
display(out_df)

In [ ]:
print("=== Outlier Decision Justifications ===")
print()
decisions = [
    ('height_cm',               'Cap',           '10 outliers (0.07%). Extreme heights likely measurement errors. Cap to IQR fence.'),
    ('weight_kg',               'Keep',          '83 outliers (0.62%). Heavy individuals exist in real populations. Retain as valid.'),
    ('body fat_%',              'Keep',          '77 outliers (0.57%). Clinically possible values. Retaining adds model learning value.'),
    ('diastolic',               'Cap',           '54 outliers (0.40%). Extreme BP values — cap to physiological fence.'),
    ('systolic',                'Cap',           '29 outliers (0.22%). Same rationale as diastolic.'),
    ('gripForce',               'Keep',          '3 outliers. Near-zero grip may represent genuine inability. Retain.'),
    ('sit and bend forward_cm', 'Already Capped','409 outliers resolved at validity step. No further action.'),
    ('broad jump_cm',           'Keep',          '57 outliers (0.43%). Zero-jump entries retained as real test failures.'),
    ('age',                     'Keep',          '0 outliers. No action needed.'),
    ('sit-ups counts',          'Keep',          '0 outliers. No action needed.'),
]
for col, action, reason in decisions:
    print(f"  [{action:<15}] {col:<30}: {reason}")

print()
# Apply caps
for col in ['height_cm', 'systolic', 'diastolic']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print("Capping applied to: height_cm, systolic, diastolic")
print(f"Final dataset shape: {df.shape}")

---
## 1.10 Correlation Analysis

Measure linear relationships between variables and identify predictors of fitness class.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_class  = LabelEncoder()
le_gender = LabelEncoder()
df['class_encoded']  = le_class.fit_transform(df['class'])    # A=0, B=1, C=2, D=3
df['gender_encoded'] = le_gender.fit_transform(df['gender'])  # F=0, M=1

print("Class encoding  :", dict(zip(le_class.classes_,  le_class.transform(le_class.classes_))))
print("Gender encoding :", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))

In [ ]:
corr_cols = NUM_COLS + ['gender_encoded', 'class_encoded']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Pearson Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with target
target_corr = corr['class_encoded'].drop('class_encoded').sort_values()
print("\nCorrelation with fitness class (class_encoded):")
print(target_corr.round(3).to_string())

print()
print("=== Key Correlation Insights ===")
print("  sit and bend forward_cm (r=-0.59): STRONGEST predictor. Flexible = better grade.")
print("  sit-ups counts          (r=-0.45): Endurance strongly linked to grade.")
print("  body fat_%              (r=+0.34): Higher fat -> worse grade.")
print("  broad jump_cm           (r=-0.26): Athletic power contributes meaningfully.")
print("  gripForce               (r=-0.14): Moderate association with fitness grade.")
print("  age                     (r=+0.07): Weak — older participants slightly lower grade.")
print("  systolic / diastolic    (r~0.04):  Very weak predictors of fitness class.")
print()
print("  Multicollinearity: height_cm & gender_encoded (r=0.70) — both retained;")
print("  they capture different constructs (physical size vs. sex-based differences).")

# Flag high correlations
high = [(corr.columns[i], corr.columns[j], round(corr.iloc[i,j],3))
        for i in range(len(corr.columns))
        for j in range(i+1, len(corr.columns))
        if abs(corr.iloc[i,j]) > 0.75 and corr.columns[i] != corr.columns[j]]
if high:
    print("\nPairs with |r| > 0.75 (multicollinearity risk):")
    for a, b, r in sorted(high, key=lambda x: -abs(x[2])):
        print(f"  {a} <-> {b} : r = {r}")

---
## 1.11 Final EDA Summary

In [ ]:
print("=" * 70)
print("  FINAL EDA SUMMARY — CLASSIFICATION NOTEBOOK")
print("=" * 70)

print()
print("--- FIVE KEY INSIGHTS ---")
insights = [
    "1. Target is near-perfectly balanced (~25% per class A/B/C/D). No SMOTE needed.",
    "2. Flexibility (sit-and-reach, r=-0.59) and endurance (sit-ups, r=-0.45) are the",
    "   strongest predictors of fitness grade — far more than cardiovascular measures.",
    "3. Height and grip force show clear bimodal distributions driven by gender.",
    "   Gender is a critical conditioning variable for multiple features.",
    "4. Body fat % has a strong positive correlation (r=+0.34) with lower fitness grade.",
    "5. Blood pressure (systolic/diastolic) adds minimal classification signal (r~0.04).",
]
for line in insights:
    print("  " + line)

print()
print("--- FIVE POTENTIAL DATA QUALITY PROBLEMS ---")
problems = [
    "1. Impossible blood pressure — 8 rows with systolic or diastolic <= 40 mmHg. Removed.",
    "2. Extreme sit-and-reach values (up to 213 cm, well beyond human limits). Capped.",
    "3. 1 exact duplicate row present in raw data. Removed.",
    "4. 10 broad jump = 0 cm entries — ambiguous (error vs. genuine failure). Kept.",
    "5. Age stored as float64 instead of int — not blocking but signals pipeline issue.",
]
for line in problems:
    print("  " + line)

print()
print("--- RECOMMENDED PREPROCESSING STEPS ---")
steps = [
    "1. Encode gender (F=0, M=1) and class (A=0 ... D=3) using LabelEncoder.",
    "2. Engineer BMI = weight / height^2 to synthesise body composition signal.",
    "3. Apply StandardScaler to all numeric features (fitted on train only).",
    "4. Use StratifiedKFold (k=5) cross-validation to preserve class ratios.",
    "5. Consider gender-stratified analysis for grip force and body fat features.",
]
for line in steps:
    print("  " + line)

print()
print(f"Dataset shape after full EDA cleaning: {df.shape}")

---
## 2. Feature Engineering

In [ ]:
# BMI feature
df['BMI'] = df['weight_kg'] / ((df['height_cm'] / 100) ** 2)
print(f"BMI created: mean={df['BMI'].mean():.2f}, std={df['BMI'].std():.2f}")

FEATURES = ['age','height_cm','weight_kg','body fat_%','diastolic','systolic',
            'gripForce','sit and bend forward_cm','sit-ups counts',
            'broad jump_cm','BMI','gender_encoded']
TARGET = 'class_encoded'

X = df[FEATURES]
y = df[TARGET]
print(f"\nFeature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"\nClass distribution:\n{y.value_counts().sort_index().rename({0:'A',1:'B',2:'C',3:'D'}).to_string()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train shape : {X_train_s.shape}")
print(f"Test shape  : {X_test_s.shape}")
print("StandardScaler fitted on train only — no data leakage into test set.")

---
## 3. Model Building & Evaluation

Three classifiers trained, evaluated, and compared.


